# Retrained transfer evaluation: excluding misgendering

Aggregates held-out test performance after task-specific retraining for the parallel outcome.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import (
    LongformerTokenizer, LongformerForSequenceClassification,
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_score, recall_score

In [ ]:
# --- Longformer Dataset ---
class LongformerTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global_attention=True, global_attention_target='cls', dynamic_padding=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.global_attention_target = global_attention_target
        self.dynamic_padding = dynamic_padding
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        padding_strategy = "longest" if self.dynamic_padding else "max_length"
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=padding_strategy, max_length=self.max_length)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)
        }
        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)
            if self.global_attention_target == 'cls':
                global_attention_mask[0] = 1
            output['global_attention_mask'] = global_attention_mask
        return output

def get_longformer_predictions(model, data_loader, device):
    model.eval()
    predictions, all_probs = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            # labels = batch['labels'].to(device)  # not used for prediction
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
            probabilities = torch.sigmoid(outputs.logits)
            predicted = (probabilities >= 0.5).float()
            predictions.extend(predicted.view(-1).cpu().numpy())
            all_probs.extend(probabilities.view(-1).cpu().numpy())
    return np.array(predictions), np.array(all_probs)

# --- Model configs: ONLY Longformer ---
longformer_model_configs = [
    {
        "name": "Longformer_GEP",
        "model_dir": str(MODEL_DIR / 'Longformer_GEP_exclude_misgender'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_mimic_GEP",
        "model_dir": str(MODEL_DIR / 'Longformer_mimic_GEP_exclude_misgender'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_berkeley_mimic_GEP",
        "model_dir": str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP_exclude_misgender'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_berkeley_phenotype_mimic_GEP",
        "model_dir": str(MODEL_DIR / 'Longformer_berkeley_phenotype_mimic_GEP_80_20_exclude_misgender'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    }
]

In [ ]:
mimic_test = pd.read_csv(str(DATA_DIR / 'GEP_test_80_20.csv'))

In [ ]:
test_texts = mimic_test['text'].tolist()
test_labels = mimic_test['label_exclude_misgendering'].tolist()

In [ ]:
# === Full Implementation (Overall + GEP=0/1, single progress bar per model, WITH SAVED PREDICTIONS) ===
import math
import numpy as np
import pandas as pd
import torch
import os
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# NEW: for BERT/ClinicalBERT
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---- Assumptions (must exist in your environment) ----
# - longformer_model_configs: list of dicts with keys ["name","model_dir","tokenizer_cls","model_cls","max_length"]
# - LongformerTextDataset: your dataset class returning dicts: 'input_ids','attention_mask', optional 'global_attention_mask'
# - get_longformer_predictions(model, loader, device)
# - test_texts, test_labels: sequences aligned with mimic_test rows
# - mimic_test: a DataFrame containing a 'GEP' column with values {0,1}
# (you already defined all of the above earlier)
# ------------------------------------------------------

BATCH_SIZE = 48
NUM_WORKERS = 0
PIN_MEMORY = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_metrics(y_true, y_pred, y_prob):
    """Compute metrics safely; AUC = NaN if only one class present."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1, "AUC": auc}

all_rows = []
all_pred_rows = []   # each row: {"Model","Row","True","Pred","Prob","GEP"}

# ==========================
# Longformer family (yours)
# ==========================
for config in longformer_model_configs:
    model_name    = config["name"]
    model_dir     = config["model_dir"]
    tokenizer_cls = config["tokenizer_cls"]
    model_cls     = config["model_cls"]
    max_length    = config["max_length"]

    print(f"\nLoading and evaluating model: {model_name}")
    try:
        tokenizer = tokenizer_cls.from_pretrained(model_dir)
        model = model_cls.from_pretrained(model_dir)
        model.to(device)
        model.eval()

        test_dataset = LongformerTextDataset(test_texts, test_labels, tokenizer, max_length=max_length)
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        )

        probs_arr = []
        preds_arr = []

        # Single progress bar per model
        pbar = tqdm(total=len(test_loader), desc=f"{model_name}: predicting", unit="batch", leave=True)
        with torch.inference_mode():
            for batch in test_loader:
                try:
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    global_attention_mask = batch.get("global_attention_mask", None)
                    if global_attention_mask is not None:
                        global_attention_mask = global_attention_mask.to(device, non_blocking=True)

                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        global_attention_mask=global_attention_mask
                    )
                    logits = outputs.logits
                    probs = torch.sigmoid(logits).view(-1).detach().cpu().numpy()
                    preds = (probs >= 0.5).astype(np.float32)

                    probs_arr.extend(probs.tolist())
                    preds_arr.extend(preds.tolist())
                except Exception as e:
                    print(f"  [Warn] Skipping a batch due to error: {e}")
                finally:
                    pbar.update(1)
        pbar.close()

        probs_arr = np.asarray(probs_arr)
        preds_arr = np.asarray(preds_arr)
        y_true_arr = np.asarray(test_labels)

        n_shared = min(len(y_true_arr), len(preds_arr), len(probs_arr), len(mimic_test))
        y_true_arr = y_true_arr[:n_shared]
        preds_arr  = preds_arr[:n_shared]
        probs_arr  = probs_arr[:n_shared]

        overall = compute_metrics(y_true_arr, preds_arr, probs_arr)
        all_rows.append({
            "Model": model_name,
            "Group": "Overall",
            "N": int(len(y_true_arr)),
            **overall
        })

        if "GEP" not in mimic_test.columns:
            print("  [Warn] 'GEP' column not found in mimic_test; skipping subgroup analysis.")
            for g in (0, 1):
                all_rows.append({
                    "Model": model_name, "Group": f"GEP={g}", "N": 0,
                    "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                    "F1 Score": np.nan, "AUC": np.nan
                })
            for i in range(n_shared):
                all_pred_rows.append({
                    "Model": model_name, "Row": int(i),
                    "True": int(y_true_arr[i]), "Pred": int(preds_arr[i]),
                    "Prob": float(probs_arr[i]), "GEP": np.nan
                })
        else:
            gep_vec = mimic_test["GEP"].to_numpy()[:n_shared]
            try:
                gep_vec = gep_vec.astype(int)
            except Exception:
                gep_vec = pd.Series(gep_vec).map(lambda x: 1 if str(x).strip().lower() in ["1","true","yes"] else 0).to_numpy()

            print(f"  Subgroup counts -> GEP=0: {(gep_vec==0).sum()}, GEP=1: {(gep_vec==1).sum()}")

            idx0 = np.where(gep_vec == 0)[0]
            if idx0.size:
                mets0 = compute_metrics(y_true_arr[idx0], preds_arr[idx0], probs_arr[idx0])
                all_rows.append({"Model": model_name, "Group": "GEP=0", "N": int(idx0.size), **mets0})
            else:
                all_rows.append({"Model": model_name, "Group": "GEP=0", "N": 0,
                                 "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                                 "F1 Score": np.nan, "AUC": np.nan})

            idx1 = np.where(gep_vec == 1)[0]
            if idx1.size:
                mets1 = compute_metrics(y_true_arr[idx1], preds_arr[idx1], probs_arr[idx1])
                all_rows.append({"Model": model_name, "Group": "GEP=1", "N": int(idx1.size), **mets1})
            else:
                all_rows.append({"Model": model_name, "Group": "GEP=1", "N": 0,
                                 "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                                 "F1 Score": np.nan, "AUC": np.nan})

            for i in range(n_shared):
                all_pred_rows.append({
                    "Model": model_name,
                    "Row": int(i),
                    "True": int(y_true_arr[i]),
                    "Pred": int(preds_arr[i]),
                    "Prob": float(probs_arr[i]),
                    "GEP": int(gep_vec[i])
                })

    except Exception as e:
        print(f"[Error] Loading or processing model {model_name}: {e}")
        all_rows.append({
            "Model": model_name,
            "Group": "Error",
            "N": np.nan,
            "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
            "F1 Score": np.nan, "AUC": np.nan,
            "Error": str(e)
        })

# ============================================================
# === Evaluate BERT & ClinicalBERT (chunked, max-pooling) ===
# ============================================================
BERT_DIR         = str(MODEL_DIR / 'BERT_GEP_exclude_misgender')
CLINICALBERT_DIR = str(MODEL_DIR / 'clinicalbert_GEP_exclude_misgender')

bert_texts  = mimic_test["text"].astype(str).tolist()
bert_labels = mimic_test["label_exclude_misgendering"].astype(int).tolist()
bert_gep    = mimic_test["GEP"].astype(int).to_numpy()

DOC_MAX_TOKENS = 4096
CHUNK_SIZE     = 510
MAX_LENGTH     = 512

class ChunkedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.sep_token

    def __len__(self):
        return len(self.texts)

    def _chunk_ids(self, text):
        token_ids = self.tokenizer.encode(text, add_special_tokens=False, truncation=False)
        token_ids = token_ids[: self.doc_max_length]
        chunks = []
        i = 0
        while i < len(token_ids):
            core = token_ids[i:i+self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + core + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk = chunk + [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            else:
                chunk = chunk[: self.max_length]
            chunks.append(chunk)
            i += self.chunk_size
        if not chunks:
            chunk = [self.tokenizer.cls_token_id, self.tokenizer.sep_token_id]
            chunk = chunk + [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunks = [chunk]
        return torch.tensor(chunks, dtype=torch.long)

    def __getitem__(self, idx):
        chunks = self._chunk_ids(self.texts[idx])
        label = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        return {"chunks": chunks, "label": label, "num_chunks": chunks.size(0)}

def bert_collate_fn(batch):
    all_chunks = [b["chunks"] for b in batch]
    labels = torch.tensor([b["label"].item() for b in batch], dtype=torch.float)
    num_chunks = [b["num_chunks"] for b in batch]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {"chunks": flat_chunks, "labels": labels, "num_chunks": num_chunks}

@torch.inference_mode()
def bertlike_predict(model, tokenizer, texts, labels, batch_size=4):
    ds = ChunkedTextDataset(texts, labels, tokenizer,
                            chunk_size=CHUNK_SIZE, max_length=MAX_LENGTH, doc_max_length=DOC_MAX_TOKENS)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=bert_collate_fn,
                    pin_memory=torch.cuda.is_available())
    all_probs, all_preds = [], []
    pbar = tqdm(dl, desc="BERT-like predicting", unit="batch", leave=True)
    for batch in pbar:
        chunks = batch["chunks"].to(device, non_blocking=True)
        num_chunks = batch["num_chunks"]
        attn = (chunks != tokenizer.pad_token_id).to(device, non_blocking=True)
        logits = model(input_ids=chunks, attention_mask=attn).logits.squeeze(-1)
        pooled_logits = []
        idx = 0
        for nc in num_chunks:
            pooled_logits.append(torch.max(logits[idx:idx+nc]))
            idx += nc
        pooled_logits = torch.stack(pooled_logits)
        probs = torch.sigmoid(pooled_logits).detach().cpu().numpy().reshape(-1)
        preds = (probs >= 0.5).astype(np.int32)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
    return np.array(all_preds), np.array(all_probs)

def eval_bertlike_model(model_dir, display_name):
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.sep_token
        model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()

        y_true = np.asarray(bert_labels)
        y_pred, y_prob = bertlike_predict(model, tokenizer, bert_texts, bert_labels, batch_size=4)

        n_shared = min(len(y_true), len(y_pred), len(y_prob), len(bert_gep))
        y_true, y_pred, y_prob, gep_vec = y_true[:n_shared], y_pred[:n_shared], y_prob[:n_shared], bert_gep[:n_shared]

        m_overall = compute_metrics(y_true, y_pred, y_prob)
        all_rows.append({"Model": display_name, "Group": "Overall", "N": int(len(y_true)), **m_overall})

        idx0 = np.where(gep_vec == 0)[0]
        if idx0.size:
            m0 = compute_metrics(y_true[idx0], y_pred[idx0], y_prob[idx0])
            all_rows.append({"Model": display_name, "Group": "GEP=0", "N": int(idx0.size), **m0})
        else:
            all_rows.append({"Model": display_name, "Group": "GEP=0", "N": 0,
                             "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                             "F1 Score": np.nan, "AUC": np.nan})

        idx1 = np.where(gep_vec == 1)[0]
        if idx1.size:
            m1 = compute_metrics(y_true[idx1], y_pred[idx1], y_prob[idx1])
            all_rows.append({"Model": display_name, "Group": "GEP=1", "N": int(idx1.size), **m1})
        else:
            all_rows.append({"Model": display_name, "Group": "GEP=1", "N": 0,
                             "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                             "F1 Score": np.nan, "AUC": np.nan})

        for i in range(n_shared):
            all_pred_rows.append({
                "Model": display_name,
                "Row": int(i),
                "True": int(y_true[i]),
                "Pred": int(y_pred[i]),
                "Prob": float(y_prob[i]),
                "GEP": int(gep_vec[i])
            })

        print(f" Evaluated {display_name}: N={n_shared}, GEP0={(gep_vec==0).sum()}, GEP1={(gep_vec==1).sum()}")

    except Exception as e:
        print(f"[Error] Evaluating {display_name}: {e}")
        all_rows.append({
            "Model": display_name, "Group": "Error", "N": np.nan,
            "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1 Score": np.nan, "AUC": np.nan,
            "Error": str(e)
        })

# Run BERT & ClinicalBERT
eval_bertlike_model(BERT_DIR, "BERT_base")
eval_bertlike_model(CLINICALBERT_DIR, "ClinicalBERT")

# ============================================================
# === Evaluate Traditional ML Models (Best in Each Family) ===
# ============================================================
import joblib

print("\n Evaluating Traditional Machine Learning Models (Best by Accuracy)")

model_dir = str(MODEL_DIR / 'GEP')

ml_model_files = {
    "SVM_tfidf": "SVM_tfidf_best_model_f1_lowercase_exclude_misgender.joblib",
    "RF_count": "RF_count_best_model_f1_lowercase_exclude_misgender.joblib",
    "LR_tfidf": "LR_tfidf_best_model_f1_lowercase_exclude_misgender.joblib",
    "NB_tfidf": "NB_tfidf_best_model_f1_lowercase_exclude_misgender.joblib",
}

val_texts = mimic_test["text"].tolist()
val_labels = mimic_test["label"].tolist()
ml_gep_vec = mimic_test["GEP"].astype(int).to_numpy()

def safe_predict_proba(model, texts):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(texts)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(texts)
    else:
        preds = model.predict(texts)
        return np.array(preds, dtype=float)

for model_name, file_name in ml_model_files.items():
    model_path = os.path.join(model_dir, file_name)
    if not os.path.exists(model_path):
        print(f"[Warning] Missing file: {model_path}")
        continue

    try:
        model = joblib.load(model_path)
        print(f"\nEvaluating {model_name}...")

        y_true = np.asarray(val_labels)
        y_pred = model.predict(val_texts)
        y_prob = safe_predict_proba(model, val_texts)

        n_shared = min(len(y_true), len(y_pred), len(y_prob), len(ml_gep_vec))
        y_true, y_pred, y_prob, gep_vec = (
            y_true[:n_shared],
            y_pred[:n_shared],
            y_prob[:n_shared],
            ml_gep_vec[:n_shared],
        )

        overall = compute_metrics(y_true, y_pred, y_prob)
        all_rows.append({
            "Model": model_name,
            "Group": "Overall",
            "N": int(len(y_true)),
            **overall
        })

        idx0 = np.where(gep_vec == 0)[0]
        if idx0.size:
            m0 = compute_metrics(y_true[idx0], y_pred[idx0], y_prob[idx0])
            all_rows.append({"Model": model_name, "Group": "GEP=0", "N": int(idx0.size), **m0})
        else:
            all_rows.append({"Model": model_name, "Group": "GEP=0", "N": 0,
                             "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                             "F1 Score": np.nan, "AUC": np.nan})

        idx1 = np.where(gep_vec == 1)[0]
        if idx1.size:
            m1 = compute_metrics(y_true[idx1], y_pred[idx1], y_prob[idx1])
            all_rows.append({"Model": model_name, "Group": "GEP=1", "N": int(idx1.size), **m1})
        else:
            all_rows.append({"Model": model_name, "Group": "GEP=1", "N": 0,
                             "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                             "F1 Score": np.nan, "AUC": np.nan})

        for i in range(n_shared):
            all_pred_rows.append({
                "Model": model_name,
                "Row": int(i),
                "True": int(y_true[i]),
                "Pred": int(y_pred[i]),
                "Prob": float(y_prob[i]),
                "GEP": int(gep_vec[i])
            })

    except Exception as e:
        print(f"[Error] Evaluating {model_name}: {e}")
        all_rows.append({
            "Model": model_name,
            "Group": "Error",
            "N": np.nan,
            "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
            "F1 Score": np.nan, "AUC": np.nan,
            "Error": str(e)
        })

# ---- Results DataFrame + Save ----
results_df = pd.DataFrame(all_rows)

metric_cols = ["Model", "Group", "N", "Accuracy", "Precision", "Recall", "F1 Score", "AUC", "Error"]
for col in metric_cols:
    if col not in results_df.columns:
        results_df[col] = np.nan
results_df = results_df[metric_cols]

metrics_csv = str(RESULTS_DIR / 'model_retrained_performance_by_GEP_exclude_misgender.csv')
metrics_xlsx = str(RESULTS_DIR / 'model_retrained_performance_by_GEP_exclude_misgender.xlsx')
results_df.to_csv(metrics_csv, index=False)
results_df.to_excel(metrics_xlsx, index=False)

preds_df = pd.DataFrame(all_pred_rows, columns=["Model","Row","True","Pred","Prob","GEP"])
preds_csv = str(RESULTS_DIR / 'GEP_Transfer_retrained_predictions_exclude_misgender.csv')
preds_xlsx = str(RESULTS_DIR / 'GEP_Transfer_retrained_predictions_exclude_misgender.xlsx')
# Per-row predictions are local restricted artifacts and must not be committed.
preds_df.to_csv(preds_csv, index=False)
preds_df.to_excel(preds_xlsx, index=False)

print("\n Results saved to:")
print(f" - {metrics_csv}")
print(f" - {metrics_xlsx}")
print(f" - {preds_csv}")
print(f" - {preds_xlsx}")

try:
    from IPython.display import display
    print("\nMetrics (head):")
    display(results_df.head(12))
    print("\nPredictions (head):")
except Exception:
    print(results_df.head(12))


In [ ]:
results_df.shape

In [ ]:
results_df

In [ ]:
import pandas as pd

# Path to the saved unified results
metrics_csv = str(RESULTS_DIR / 'model_retrained_performance_by_GEP_exclude_misgender.csv')

# Load
df = pd.read_csv(metrics_csv)

# Filter for Overall group only
overall_df = df[df["Group"] == "Overall"].copy()

# Drop any models with missing F1 or Accuracy
overall_df = overall_df.dropna(subset=["Accuracy", "F1 Score"])

# Rank by F1 (you can switch to Accuracy if you prefer)
overall_df["Rank_F1"] = overall_df["F1 Score"].rank(ascending=False, method="dense").astype(int)
overall_df["Rank_Accuracy"] = overall_df["Accuracy"].rank(ascending=False, method="dense").astype(int)

# Sort by F1 rank
overall_df = overall_df.sort_values(by=["Rank_F1", "Rank_Accuracy"])

# Keep key columns
summary = overall_df[[
    "Rank_F1", "Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC"
]]

# Format nicely
summary = summary.reset_index(drop=True)
summary.index += 1

print("\n Model Ranking by F1 Score (Overall Performance):")
print(summary.to_string(index=False, justify="center", col_space=10))

# Optionally save for paper figures
summary_path = str(RESULTS_DIR / 'model_ranking_summary_GEP_exclude_misgender.csv')
summary.to_csv(summary_path, index=False)
print(f"\n Summary table saved to {summary_path}")


In [ ]:
# ============================
# EVAL: subcategory-by-GEP + save per-row predictions
# N_category = count of that subtype in that subgroup
# ============================
import os
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Configs
BATCH_SIZE   = 48
NUM_WORKERS  = 0
PIN_MEMORY   = False
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR  = str(RESULTS_DIR) + os.sep
os.makedirs(RESULTS_DIR, exist_ok=True)

TARGET_COLS  = ['label_exclude_misgendering', 'Credibility and Obstinacy', 'Compliance', 'Descriptors', 'Misgendering']
REQUIRED_COLS = ['text', 'GEP'] + TARGET_COLS

# Guards & normalization
_missing = [c for c in REQUIRED_COLS if c not in mimic_test.columns]
if _missing:
    raise ValueError(f"mimic_test is missing required columns: {_missing}")

for c in ['GEP'] + TARGET_COLS:
    mimic_test[c] = pd.to_numeric(mimic_test[c], errors='coerce').fillna(0).astype(int)

# Helpers
def _safe_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        return float("nan")

def _metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1 Score":  f1_score(y_true, y_pred, zero_division=0),
        "AUC":       _safe_auc(y_true, y_prob),
    }

all_rows = []
all_pred_rows = []

for cfg in longformer_model_configs:
    model_name    = cfg["name"]
    model_dir     = cfg["model_dir"]
    tokenizer_cls = cfg["tokenizer_cls"]
    model_cls     = cfg["model_cls"]
    max_length    = cfg.get("max_length", 4096)
    use_global_attention = cfg.get("use_global_attention", False)

    print(f"\nLoading and evaluating model: {model_name}")
    try:
        # Load model + tokenizer
        tokenizer = tokenizer_cls.from_pretrained(model_dir)
        model = model_cls.from_pretrained(model_dir).to(DEVICE).eval()

        # Build dataset/loader
        test_dataset = LongformerTextDataset(
            texts=test_texts,
            labels=test_labels,
            tokenizer=tokenizer,
            max_length=max_length,
            use_global_attention=use_global_attention,
            global_attention_target='cls',
            dynamic_padding=False
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        )

        preds_arr, probs_arr = get_longformer_predictions(model, test_loader, DEVICE)

        # Align lengths
        n_shared = min(len(preds_arr), len(probs_arr), len(mimic_test))
        preds_arr = preds_arr[:n_shared].astype(int)
        probs_arr = probs_arr[:n_shared].astype(float)

        truths = {c: mimic_test[c].to_numpy()[:n_shared] for c in TARGET_COLS}
        gep    = mimic_test['GEP'].to_numpy()[:n_shared]

        # helper: now we separate "how many we evaluated on" vs "how many positives"
        def _append_row(model_display, group, y_true, y_pred, y_prob, err=None):
            m = _metrics(y_true, y_pred, y_prob)
            all_rows.append({
                "Model": model_display,
                "Group": group,
                # <- what you want:
                # number of rows in THIS group/category that actually have this subtype
                "N_category": int(y_true.sum()),
                # <- how many rows were evaluated to get the metrics (the denominator)
                "N_eval": int(len(y_true)),
                "Accuracy": m["Accuracy"],
                "Precision": m["Precision"],
                "Recall": m["Recall"],
                "F1 Score": m["F1 Score"],
                "AUC": m["AUC"],
                "Error": np.nan if err is None else str(err),
            })

        for tgt in TARGET_COLS:
            model_display = f"{model_name} [{tgt}]"

            # ---------- Overall ----------
            y_true_all = truths[tgt]
            _append_row(model_display, "Overall",
                        y_true_all, preds_arr, probs_arr)

            # ---------- GEP=0 / GEP=1 ----------
            for g in (0, 1):
                idx = np.where(gep == g)[0]
                if idx.size == 0:
                    all_rows.append({
                        "Model": model_display,
                        "Group": f"GEP={g}",
                        "N_category": 0,
                        "N_eval": 0,
                        "Accuracy": np.nan,
                        "Precision": np.nan,
                        "Recall": np.nan,
                        "F1 Score": np.nan,
                        "AUC": np.nan,
                        "Error": np.nan,
                    })
                else:
                    y_true_g = truths[tgt][idx]      # e.g. all Cred&Obs for GEP=0
                    y_pred_g = preds_arr[idx]
                    y_prob_g = probs_arr[idx]
                    _append_row(model_display, f"GEP={g}",
                                y_true_g, y_pred_g, y_prob_g)

        # per-row predictions
        for i in range(n_shared):
            all_pred_rows.append({
                "Model": model_name,
                "Row": int(i),
                "Prob": float(probs_arr[i]),
                "Pred": int(preds_arr[i]),
                "GEP": int(gep[i]),
                "True_label": int(truths['label_exclude_misgendering'][i]),
                "True_Credibility and Obstinacy": int(truths['Credibility and Obstinacy'][i]),
                "True_Compliance": int(truths['Compliance'][i]),
                "True_Descriptors": int(truths['Descriptors'][i]),
                # (you can add Misgendering here too if needed)
            })

    except Exception as e:
        print(f"[Error] Loading or processing model {model_name}: {e}")
        all_rows.append({
            "Model": model_name,
            "Group": "Error",
            "N_category": np.nan,
            "N_eval": np.nan,
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1 Score": np.nan,
            "AUC": np.nan,
            "Error": str(e)
        })

# ------------------------------
# Save results (same filenames)
# ------------------------------
results_df = pd.DataFrame(all_rows)
cols = ["Model","Group","N_category","N_eval",
        "Accuracy","Precision","Recall","F1 Score","AUC","Error"]
for c in cols:
    if c not in results_df.columns: results_df[c] = np.nan
results_df = results_df[cols]

metrics_csv  = os.path.join(RESULTS_DIR, "subcategory_and_label_metrics_by_GEP_exclude_misgender.csv")
metrics_xlsx = os.path.join(RESULTS_DIR, "subcategory_and_label_metrics_by_GEP_exclude_misgender.xlsx")
results_df.to_csv(metrics_csv, index=False)
results_df.to_excel(metrics_xlsx, index=False)

preds_df = pd.DataFrame(all_pred_rows, columns=[
    "Model","Row","Prob","Pred","GEP",
    "True_label","True_Credibility and Obstinacy","True_Compliance","True_Descriptors"
])
preds_csv  = os.path.join(RESULTS_DIR, "per_row_predictions_all_models_exclude_misgender.csv")
preds_xlsx = os.path.join(RESULTS_DIR, "per_row_predictions_all_models_exclude_misgender.xlsx")
# Per-row predictions are local restricted artifacts and must not be committed.
preds_df.to_csv(preds_csv, index=False)
preds_df.to_excel(preds_xlsx, index=False)

print("\n Results saved to:")
print(f" - {metrics_csv}")
print(f" - {metrics_xlsx}")
print(f" - {preds_csv}")
print(f" - {preds_xlsx}")

try:
    from IPython.display import display
    print("\nMetrics (head):")
    display(results_df.head(12))
    print("\nPredictions (head):")
except Exception:
    print(results_df.head(12))
